# Supplementary repeated stratified 5×5 CV stability analysis

This notebook runs a supplementary stability analysis for the final screening-oriented modeling strategy.

It does **not** replace the final held-out 75/25 results. It assesses whether the same broad pipeline remains stable across alternative stratified partitions.

Key safeguards:

- Scaling is fitted inside each training fold only.
- PCA is fitted inside each training fold only.
- PCA component retention is applied inside each training fold.
- Fold-level metrics are computed on the corresponding validation fold.
- The analysis reports mean, SD, median, and empirical 2.5th/97.5th percentiles across 5×5 = 25 partitions.

Final strategy approximated here:

- Victimization: cost-complexity pruned decision tree. The pruning alpha is selected inside the outer training fold using an internal validation split.
- Perpetration: compact deterministic feed-forward neural network with class weighting and early stopping.
- Victim–perpetrator overlap: weighted logistic regression, positive-class sample-weight multiplier = 1.5.

Important manuscript wording: this is a **supplementary stability check**, not nested model selection and not a replacement for external validation.


In [1]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import os
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd().resolve()
OUTPUT_DIR = BASE_DIR / "final_repeated_cv_stability_PCA_trainonly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
PCA_THRESHOLD = 0.95
THRESHOLD = 0.50

# Perpetration DNN settings. Reduce EPOCHS for quick smoke tests; use 80-120 for final.
RUN_PERPETRATION_DNN = True
DNN_EPOCHS = 100
DNN_BATCH_SIZE = 64
DNN_PATIENCE = 12
DNN_POS_WEIGHT_MULTIPLIER = 1.0

# Victimization tree internal pruning selection.
TREE_INNER_VALIDATION_SIZE = 0.20
TREE_ALPHA_MAX_CANDIDATES = 50
TREE_RECALL_TARGET = 0.85

# Overlap logistic sample weighting.
OVERLAP_POS_WEIGHT_MULTIPLIER = 1.5

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


BASE_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final
OUTPUT_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_repeated_cv_stability_PCA_trainonly


In [2]:
# ============================================================
# DATA LOADING AND ANALYTIC MATRIX CONSTRUCTION
# ============================================================

def find_data_dir(start: Path) -> Path:
    candidates = []
    p = start.resolve()
    for _ in range(5):
        candidates.append(p / "data")
        p = p.parent
    for c in candidates:
        if (c / "lista_global_vars.csv").exists() and (c / "target_col.csv").exists():
            return c
    raise FileNotFoundError("Could not find data/lista_global_vars.csv and data/target_col.csv")

DATA_DIR = find_data_dir(BASE_DIR)
FEATURES_PATH = DATA_DIR / "lista_global_vars.csv"
TARGET_PATH = DATA_DIR / "target_col.csv"

feat_df = pd.read_csv(FEATURES_PATH)
target_df = pd.read_csv(TARGET_PATH).fillna(0)

print("DATA_DIR:", DATA_DIR)
print("feat_df:", feat_df.shape)
print("target_df:", target_df.shape)
print("feature columns:", list(feat_df.columns))
print("target columns:", list(target_df.columns))

V_COL = "V.SUM.TOTAL"
P_COL = "P.SUM.TOTAL"
TARGET_DROP_COLS = [
    "VÍCTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR", "POLIVICTIMIZACION",
    "POLIPERPETRACION", "SOLO.VICTIMA", "SOLO.PERPETRADOR", "NO.VICT_NO.PERP",
    "V.O", "P.SUM.TOTAL", "V.SUM.TOTAL"
]

if V_COL not in target_df.columns or P_COL not in target_df.columns:
    raise ValueError("target_col.csv must contain V.SUM.TOTAL and P.SUM.TOTAL")

df_merged = feat_df.join(target_df, how="inner")

rare_mask = pd.Series(False, index=df_merged.index)
if "GENERO_BIN_2" in df_merged.columns:
    rare_mask = rare_mask | (pd.to_numeric(df_merged["GENERO_BIN_2"], errors="coerce") == 1)
if "ORIENTSEX.BN_3" in df_merged.columns:
    rare_mask = rare_mask | (pd.to_numeric(df_merged["ORIENTSEX.BN_3"], errors="coerce") == 1)

print("Rows removed by rare category filter:", int(rare_mask.sum()))
df_merged = df_merged.loc[~rare_mask].copy().reset_index(drop=True)

for c in ["GENERO_BIN_2", "ORIENTSEX.BN_3"]:
    if c in df_merged.columns:
        df_merged = df_merged.drop(columns=[c])

df_merged[V_COL] = pd.to_numeric(df_merged[V_COL], errors="coerce").fillna(0)
df_merged[P_COL] = pd.to_numeric(df_merged[P_COL], errors="coerce").fillna(0)

y_victim = (df_merged[V_COL] >= 1).astype(int)
y_perp = (df_merged[P_COL] >= 1).astype(int)
y_overlap = ((df_merged[V_COL] >= 1) & (df_merged[P_COL] >= 1)).astype(int)

df_features = df_merged.drop(columns=[c for c in TARGET_DROP_COLS if c in df_merged.columns], errors="ignore").copy()
for c in ["INTERSECT", "victim_count_ge1", "perp_count_ge1", "overlap_count_ge1"]:
    if c in df_features.columns:
        df_features = df_features.drop(columns=[c])

for c, mapping in {
    "PAÍS": {1: True, 2: False},
    "ETNIA.BN": {0.0: False, 1.0: True},
    "FUGAS.BN": {0.0: False, 1.0: True},
}.items():
    if c in df_features.columns:
        df_features[c] = df_features[c].replace(mapping)

if "GENERO_BIN_0" in df_features.columns:
    df_features["GENERO.BN0"] = df_features["GENERO_BIN_0"].replace({0.0: False, 1.0: True})
if "GENERO_BIN_1" in df_features.columns:
    df_features["GENERO.BN1"] = df_features["GENERO_BIN_1"].replace({0.0: False, 1.0: True})
if "ORIENTSEX.BN_1" in df_features.columns:
    df_features["ORIENTSEX.BN0"] = df_features["ORIENTSEX.BN_1"].replace({0.0: False, 1.0: True})
if "ORIENTSEX.BN_2" in df_features.columns:
    df_features["ORIENTSEX.BN1"] = df_features["ORIENTSEX.BN_2"].replace({0.0: False, 1.0: True})

for c in ["GENERO_BIN_0", "GENERO_BIN_1", "ORIENTSEX.BN_1", "ORIENTSEX.BN_2"]:
    if c in df_features.columns:
        df_features = df_features.drop(columns=[c])

if "CONVIVEN.5" in df_features.columns:
    df_features = df_features.rename(columns={"CONVIVEN.5": "CONVIVEN_H"})
    df_features["CONVIVEN_H"] = df_features["CONVIVEN_H"].replace({0.0: False, 1.0: True})
if "CONVIVEN.6" in df_features.columns:
    df_features = df_features.rename(columns={"CONVIVEN.6": "CONVIVEN_0"})
    df_features["CONVIVEN_0"] = df_features["CONVIVEN_0"].replace({0.0: False, 1.0: True})

for c in df_features.columns:
    if df_features[c].dtype == "bool":
        df_features[c] = df_features[c].astype(int)
    else:
        df_features[c] = pd.to_numeric(df_features[c], errors="coerce")

missing_before = int(df_features.isna().sum().sum())
if missing_before > 0:
    print("WARNING: missing predictor values filled with 0:", missing_before)
    df_features = df_features.fillna(0)

X = df_features.astype(float)
outcomes = {
    "victimization": y_victim.astype(int).reset_index(drop=True),
    "perpetration": y_perp.astype(int).reset_index(drop=True),
    "overlap": y_overlap.astype(int).reset_index(drop=True),
}

print("Analytical feature matrix:", X.shape)
print("Predictor columns:", list(X.columns))
for name, y in outcomes.items():
    print(name, "positives:", int(y.sum()), "/", len(y), "prevalence:", round(float(y.mean()), 4))

X.to_csv(OUTPUT_DIR / "stability_analysis_feature_matrix.csv", index=False)
pd.DataFrame({k: v for k, v in outcomes.items()}).to_csv(OUTPUT_DIR / "stability_analysis_targets.csv", index=False)
with open(OUTPUT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(X.columns), f, ensure_ascii=False, indent=2)


DATA_DIR: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data
feat_df: (4024, 29)
target_df: (4024, 11)
feature columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', 'CONVIVEN.5', 'CONVIVEN.6', 'AUTOEFIC.MEAN', 'AUTOEFIC.VAR', 'IMPULS.MEAN', 'IMPULS.MEDIAN', 'IMPULS.VAR', 'APOYO.MEAN', 'APOYO.MEDIAN', 'APOYO.VAR', 'MORAL.MEAN', 'MORAL.VAR', 'PORNO.T', 'GENERO_BIN_0', 'GENERO_BIN_1', 'GENERO_BIN_2', 'ORIENTSEX.BN_1', 'ORIENTSEX.BN_2', 'ORIENTSEX.BN_3']
target columns: ['VÍCTIMA', 'PERPETRADOR', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL']
Rows removed by rare category filter: 257
Analytical feature matrix: (3767, 27)
Predictor columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', '

In [3]:
# ============================================================
# METRICS AND MODEL HELPERS
# ============================================================

def fit_fold_pca(X_train_df, X_test_df, threshold=PCA_THRESHOLD):
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_df.values)
    X_test_scaled = scaler.transform(X_test_df.values)
    means = X_train_scaled.mean(axis=0)
    X_train_centered = X_train_scaled - means
    X_test_centered = X_test_scaled - means
    pca = PCA(n_components=X_train_df.shape[1], random_state=RANDOM_STATE)
    X_train_pca_all = pca.fit_transform(X_train_centered)
    X_test_pca_all = pca.transform(X_test_centered)
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n_components = int(np.argmax(cum_var >= threshold) + 1)
    return X_train_pca_all[:, :n_components], X_test_pca_all[:, :n_components], n_components, float(cum_var[n_components - 1])


def binary_metrics_from_prob(y_true, y_prob, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan
    return {
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "recall_sensitivity": float(recall), "specificity": float(specificity),
        "precision_ppv": float(ppv), "npv": float(npv),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_positive": float(f1) if pd.notna(f1) else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc) if pd.notna(roc_auc) else np.nan,
        "pr_auc_average_precision": float(pr_auc) if pd.notna(pr_auc) else np.nan,
        "positive_support": int(np.sum(y_true == 1)),
        "negative_support": int(np.sum(y_true == 0)),
        "support": int(len(y_true)),
    }


def select_pruned_tree_alpha(X_train, y_train, seed):
    y_train = np.asarray(y_train).astype(int)
    inner_train_idx, inner_val_idx = train_test_split(np.arange(len(y_train)), test_size=TREE_INNER_VALIDATION_SIZE, random_state=seed, stratify=y_train)
    X_inner_train, y_inner_train = X_train[inner_train_idx], y_train[inner_train_idx]
    X_inner_val, y_inner_val = X_train[inner_val_idx], y_train[inner_val_idx]
    base_tree = DecisionTreeClassifier(random_state=seed, class_weight=None)
    path = base_tree.cost_complexity_pruning_path(X_inner_train, y_inner_train)
    alphas = np.unique(path.ccp_alphas)
    alphas = alphas[np.isfinite(alphas)]
    alphas = alphas[alphas >= 0]
    if len(alphas) > TREE_ALPHA_MAX_CANDIDATES:
        alphas = np.unique(np.quantile(alphas, np.linspace(0, 1, TREE_ALPHA_MAX_CANDIDATES)))
    rows = []
    for alpha in alphas:
        clf = DecisionTreeClassifier(random_state=seed, ccp_alpha=float(alpha))
        clf.fit(X_inner_train, y_inner_train)
        prob = clf.predict_proba(X_inner_val)[:, 1]
        m = binary_metrics_from_prob(y_inner_val, prob, threshold=THRESHOLD)
        rows.append({"ccp_alpha": float(alpha), **m})
    cand = pd.DataFrame(rows)
    if cand.empty:
        return 0.0, cand
    eligible = cand[(cand["recall_sensitivity"] >= TREE_RECALL_TARGET) & (cand["specificity"] > 0)]
    if len(eligible) > 0:
        best = eligible.sort_values(["balanced_accuracy", "recall_sensitivity", "specificity"], ascending=[False, False, False]).iloc[0]
    else:
        nonzero = cand[cand["specificity"] > 0]
        if len(nonzero) > 0:
            best = nonzero.sort_values(["recall_sensitivity", "balanced_accuracy"], ascending=[False, False]).iloc[0]
        else:
            best = cand.sort_values("recall_sensitivity", ascending=False).iloc[0]
    return float(best["ccp_alpha"]), cand


def fit_predict_victim_tree(X_train_pca, X_test_pca, y_train, seed):
    alpha, alpha_table = select_pruned_tree_alpha(X_train_pca, y_train, seed)
    clf = DecisionTreeClassifier(random_state=seed, ccp_alpha=alpha)
    clf.fit(X_train_pca, y_train)
    return clf.predict_proba(X_test_pca)[:, 1], {"selected_ccp_alpha": alpha, "inner_alpha_table": alpha_table}


def fit_predict_overlap_logreg(X_train_pca, X_test_pca, y_train, seed):
    y_train = np.asarray(y_train).astype(int)
    pos_rate = y_train.mean()
    base_pos_weight = (1.0 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
    sample_weight = np.where(y_train == 1, base_pos_weight * OVERLAP_POS_WEIGHT_MULTIPLIER, 1.0)
    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed)
    model.fit(X_train_pca, y_train, sample_weight=sample_weight)
    return model.predict_proba(X_test_pca)[:, 1], {"positive_class_weight": float(base_pos_weight * OVERLAP_POS_WEIGHT_MULTIPLIER)}


def fit_predict_perp_dnn(X_train_pca, X_test_pca, y_train, seed):
    if not RUN_PERPETRATION_DNN:
        y_train = np.asarray(y_train).astype(int)
        pos_rate = y_train.mean()
        base_pos_weight = (1 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
        sw = np.where(y_train == 1, base_pos_weight, 1.0)
        model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed)
        model.fit(X_train_pca, y_train, sample_weight=sw)
        return model.predict_proba(X_test_pca)[:, 1], {"fallback": "logistic_regression"}

    import tensorflow as tf
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    y_train = np.asarray(y_train).astype(int)
    pos_rate = y_train.mean()
    base_pos_weight = (1 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
    class_weight = {0: 1.0, 1: float(base_pos_weight * DNN_POS_WEIGHT_MULTIPLIER)}
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_pca.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dropout(0.15),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=[tf.keras.metrics.Recall(name="recall")])
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_recall", mode="max", patience=DNN_PATIENCE, restore_best_weights=True, verbose=0)]
    hist = model.fit(X_train_pca, y_train, epochs=DNN_EPOCHS, batch_size=DNN_BATCH_SIZE, validation_split=0.20, class_weight=class_weight, callbacks=callbacks, verbose=0)
    y_prob = model.predict(X_test_pca, verbose=0).reshape(-1)
    return y_prob, {"class_weight_positive": class_weight[1], "epochs_ran": int(len(hist.history.get("loss", [])))}


In [6]:
# ============================================================
# RUN REPEATED STRATIFIED 5×5 CV
# ============================================================

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
fold_rows = []
alpha_rows = []

for outcome_name, y_series in outcomes.items():
    y = y_series.to_numpy().astype(int)
    print("" + "=" * 80)
    print("Outcome:", outcome_name, "n=", len(y), "positives=", int(y.sum()))
    for fold_id, (train_idx, test_idx) in enumerate(rskf.split(X, y), start=1):
        seed = RANDOM_STATE + fold_id
        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]
        X_train_pca, X_test_pca, n_components, retained_var = fit_fold_pca(X_train_df, X_test_df, threshold=PCA_THRESHOLD)
        if outcome_name == "victimization":
            y_prob, info = fit_predict_victim_tree(X_train_pca, X_test_pca, y_train, seed)
            model_label = "pruned_decision_tree_inner_alpha"
            if isinstance(info.get("inner_alpha_table"), pd.DataFrame):
                tmp = info["inner_alpha_table"].copy()
                tmp.insert(0, "outcome", outcome_name)
                tmp.insert(1, "fold_id", fold_id)
                alpha_rows.append(tmp)
        elif outcome_name == "perpetration":
            y_prob, info = fit_predict_perp_dnn(X_train_pca, X_test_pca, y_train, seed)
            model_label = "compact_dnn_class_weighted" if RUN_PERPETRATION_DNN else "logreg_fallback_smoke_test"
        elif outcome_name == "overlap":
            y_prob, info = fit_predict_overlap_logreg(X_train_pca, X_test_pca, y_train, seed)
            model_label = "weighted_logistic_regression_SW_pos1.5"
        else:
            raise ValueError(outcome_name)
        m = binary_metrics_from_prob(y_test, y_prob, threshold=THRESHOLD)
        row = {
            "outcome": outcome_name,
            "fold_id": fold_id,
            "model_label": model_label,
            "threshold": THRESHOLD,
            "n_components": int(n_components),
            "retained_variance": float(retained_var),
            "train_n": int(len(train_idx)),
            "test_n": int(len(test_idx)),
            "train_positive_prevalence": float(np.mean(y_train)),
            "test_positive_prevalence": float(np.mean(y_test)),
            **m,
        }
        for k, v in info.items():
            if isinstance(v, (int, float, str, bool)) or v is None:
                row[k] = v
        fold_rows.append(row)
        print(f"{outcome_name:14s} fold {fold_id:02d} | PCs={n_components:02d} var={retained_var:.3f} | recall={m['recall_sensitivity']:.3f} spec={m['specificity']:.3f} PPV={m['precision_ppv']:.3f} balacc={m['balanced_accuracy']:.3f} ROC={m['roc_auc']:.3f} PR={m['pr_auc_average_precision']:.3f}")

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(OUTPUT_DIR / "repeated_5x5_cv_fold_metrics.csv", index=False)
if alpha_rows:
    alpha_table = pd.concat(alpha_rows, ignore_index=True)
    alpha_table.to_csv(OUTPUT_DIR / "victimization_inner_pruning_alpha_tables.csv", index=False)
print("Saved fold metrics:", OUTPUT_DIR / "repeated_5x5_cv_fold_metrics.csv")
print(fold_metrics.head())


Outcome: victimization n= 3767 positives= 1861
victimization  fold 01 | PCs=18 var=0.951 | recall=0.648 spec=0.636 PPV=0.634 balacc=0.642 ROC=0.697 PR=0.666
victimization  fold 02 | PCs=19 var=0.963 | recall=0.721 spec=0.528 PPV=0.599 balacc=0.624 ROC=0.676 PR=0.646
victimization  fold 03 | PCs=18 var=0.951 | recall=0.656 spec=0.659 PPV=0.652 balacc=0.657 ROC=0.702 PR=0.676
victimization  fold 04 | PCs=18 var=0.951 | recall=0.624 spec=0.656 PPV=0.639 balacc=0.640 ROC=0.690 PR=0.652
victimization  fold 05 | PCs=18 var=0.950 | recall=0.691 spec=0.507 PPV=0.578 balacc=0.599 ROC=0.643 PR=0.636
victimization  fold 06 | PCs=18 var=0.951 | recall=0.556 spec=0.628 PPV=0.593 balacc=0.592 ROC=0.600 PR=0.575
victimization  fold 07 | PCs=18 var=0.950 | recall=0.574 spec=0.672 PPV=0.631 balacc=0.623 ROC=0.665 PR=0.635
victimization  fold 08 | PCs=18 var=0.951 | recall=0.527 spec=0.677 PPV=0.614 balacc=0.602 ROC=0.645 PR=0.627
victimization  fold 09 | PCs=18 var=0.951 | recall=0.613 spec=0.682 PPV=0

E0000 00:00:1783664459.494179 13851495 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


perpetration   fold 01 | PCs=18 var=0.951 | recall=0.712 spec=0.690 PPV=0.413 balacc=0.701 ROC=0.742 PR=0.476
perpetration   fold 02 | PCs=18 var=0.950 | recall=0.678 spec=0.310 PPV=0.232 balacc=0.494 ROC=0.515 PR=0.273
perpetration   fold 03 | PCs=18 var=0.950 | recall=0.565 spec=0.615 PPV=0.311 balacc=0.590 ROC=0.625 PR=0.345
perpetration   fold 04 | PCs=18 var=0.951 | recall=0.571 spec=0.681 PPV=0.354 balacc=0.626 ROC=0.678 PR=0.420


E0000 00:00:1783664464.525315 13851495 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


perpetration   fold 05 | PCs=18 var=0.950 | recall=0.508 spec=0.719 PPV=0.357 balacc=0.614 ROC=0.676 PR=0.434
perpetration   fold 06 | PCs=18 var=0.950 | recall=0.599 spec=0.633 PPV=0.333 balacc=0.616 ROC=0.681 PR=0.387
perpetration   fold 07 | PCs=18 var=0.951 | recall=0.548 spec=0.634 PPV=0.315 balacc=0.591 ROC=0.632 PR=0.345
perpetration   fold 08 | PCs=18 var=0.951 | recall=0.565 spec=0.484 PPV=0.252 balacc=0.525 ROC=0.543 PR=0.286
perpetration   fold 09 | PCs=18 var=0.951 | recall=0.638 spec=0.684 PPV=0.383 balacc=0.661 ROC=0.709 PR=0.441


...

perpetration   fold 14 | PCs=19 var=0.963 | recall=0.593 spec=0.635 PPV=0.333 balacc=0.614 ROC=0.665 PR=0.401
perpetration   fold 15 | PCs=18 var=0.951 | recall=0.616 spec=0.675 PPV=0.368 balacc=0.646 ROC=0.705 PR=0.433
perpetration   fold 16 | PCs=18 var=0.951 | recall=0.672 spec=0.440 PPV=0.269 balacc=0.556 ROC=0.568 PR=0.301
perpetration   fold 17 | PCs=18 var=0.951 | recall=0.588 spec=0.530 PPV=0.277 balacc=0.559 ROC=0.587 PR=0.327


E0000 00:00:1783664480.222002 13851495 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


perpetration   fold 18 | PCs=19 var=0.963 | recall=0.644 spec=0.714 PPV=0.409 balacc=0.679 ROC=0.737 PR=0.474
perpetration   fold 19 | PCs=18 var=0.951 | recall=0.605 spec=0.651 PPV=0.347 balacc=0.628 ROC=0.679 PR=0.405
perpetration   fold 20 | PCs=18 var=0.951 | recall=0.791 spec=0.306 PPV=0.259 balacc=0.548 ROC=0.620 PR=0.360
perpetration   fold 21 | PCs=18 var=0.950 | recall=0.616 spec=0.579 PPV=0.310 balacc=0.597 ROC=0.638 PR=0.354
perpetration   fold 22 | PCs=18 var=0.951 | recall=0.678 spec=0.536 PPV=0.309 balacc=0.607 ROC=0.659 PR=0.393


E0000 00:00:1783664485.427303 13851495 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


perpetration   fold 23 | PCs=18 var=0.951 | recall=0.661 spec=0.616 PPV=0.346 balacc=0.639 ROC=0.688 PR=0.385
perpetration   fold 24 | PCs=18 var=0.950 | recall=0.571 spec=0.648 PPV=0.332 balacc=0.609 ROC=0.654 PR=0.405
perpetration   fold 25 | PCs=18 var=0.950 | recall=0.689 spec=0.467 PPV=0.284 balacc=0.578 ROC=0.629 PR=0.325
Outcome: overlap n= 3767 positives= 713
overlap        fold 01 | PCs=18 var=0.951 | recall=0.776 spec=0.527 PPV=0.278 balacc=0.652 ROC=0.743 PR=0.413
overlap        fold 02 | PCs=18 var=0.951 | recall=0.755 spec=0.548 PPV=0.281 balacc=0.652 ROC=0.746 PR=0.403
overlap        fold 03 | PCs=18 var=0.950 | recall=0.824 spec=0.550 PPV=0.298 balacc=0.687 ROC=0.759 PR=0.432
overlap        fold 04 | PCs=18 var=0.950 | recall=0.761 spec=0.555 PPV=0.284 balacc=0.658 ROC=0.742 PR=0.438
overlap        fold 05 | PCs=18 var=0.950 | recall=0.818 spec=0.541 PPV=0.295 balacc=0.680 ROC=0.744 PR=0.416
overlap        fold 06 | PCs=18 var=0.950 | recall=0.783 spec=0.555 PPV=0.292 ba

In [9]:
# ============================================================
# SUMMARIZE CV STABILITY RESULTS
# ============================================================

metric_cols = ["recall_sensitivity", "specificity", "precision_ppv", "npv", "balanced_accuracy", "f1_positive", "accuracy", "roc_auc", "pr_auc_average_precision", "n_components", "retained_variance"]
summary_rows = []
for outcome, sub in fold_metrics.groupby("outcome"):
    for metric in metric_cols:
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna().to_numpy()
        summary_rows.append({
            "outcome": outcome,
            "metric": metric,
            "n_partitions": int(len(vals)),
            "mean": float(np.mean(vals)),
            "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "median": float(np.median(vals)),
            "p2_5": float(np.percentile(vals, 2.5)),
            "p97_5": float(np.percentile(vals, 97.5)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "repeated_5x5_cv_summary_long.csv", index=False)
pretty_metrics = ["recall_sensitivity", "specificity", "precision_ppv", "npv", "balanced_accuracy", "f1_positive", "accuracy", "roc_auc", "pr_auc_average_precision"]
pretty = summary[summary["metric"].isin(pretty_metrics)].copy()
pretty["formatted_mean_sd"] = pretty.apply(lambda r: f"{r['mean']*100:.1f}% ({r['sd']*100:.1f})" if pd.notna(r["sd"]) else f"{r['mean']*100:.1f}%", axis=1)
pretty["formatted_p2_5_p97_5"] = pretty.apply(lambda r: f"{r['mean']*100:.1f}% ({r['p2_5']*100:.1f}–{r['p97_5']*100:.1f})", axis=1)
pretty_wide_mean_sd = pretty.pivot(index="outcome", columns="metric", values="formatted_mean_sd").reset_index()
pretty_wide_interval = pretty.pivot(index="outcome", columns="metric", values="formatted_p2_5_p97_5").reset_index()
ordered_cols = ["outcome", "recall_sensitivity", "specificity", "precision_ppv", "npv", "balanced_accuracy", "f1_positive", "accuracy", "roc_auc", "pr_auc_average_precision"]
pretty_wide_mean_sd = pretty_wide_mean_sd[ordered_cols]
pretty_wide_interval = pretty_wide_interval[ordered_cols]
pretty_wide_mean_sd.to_csv(OUTPUT_DIR / "repeated_5x5_cv_pretty_mean_sd.csv", index=False)
pretty_wide_interval.to_csv(OUTPUT_DIR / "repeated_5x5_cv_pretty_mean_p2_5_p97_5.csv", index=False)
pca_summary = summary[summary["metric"].isin(["n_components", "retained_variance"])].copy()
pca_summary.to_csv(OUTPUT_DIR / "repeated_5x5_cv_pca_retention_summary.csv", index=False)
print("=== REPEATED 5×5 CV SUMMARY — MEAN (SD) ===")
print(pretty_wide_mean_sd.to_string(index=False))
print("=== REPEATED 5×5 CV SUMMARY — MEAN (2.5th–97.5th percentile) ===")
print(pretty_wide_interval.to_string(index=False))
print("=== PCA RETENTION SUMMARY ===")
print(pca_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
readme = f"""Supplementary repeated stratified 5x5 CV stability analysis.

This analysis does not replace the final held-out evaluation. It assesses metric stability across alternative partitions.

Configuration:
- N_SPLITS = {N_SPLITS}
- N_REPEATS = {N_REPEATS}
- PCA_THRESHOLD = {PCA_THRESHOLD}
- THRESHOLD = {THRESHOLD}
- Scaling/PCA fitted within each training fold only.
- Victimization: pruned decision tree with ccp_alpha selected inside each outer training fold.
- Perpetration: {'compact DNN with class weighting and early stopping' if RUN_PERPETRATION_DNN else 'logistic regression fallback; do not use fallback for manuscript'}.
- Overlap: weighted logistic regression, SW_pos1.5.

Main outputs:
- repeated_5x5_cv_fold_metrics.csv
- repeated_5x5_cv_summary_long.csv
- repeated_5x5_cv_pretty_mean_sd.csv
- repeated_5x5_cv_pretty_mean_p2_5_p97_5.csv
"""
(OUTPUT_DIR / "README_repeated_cv_stability.txt").write_text(readme, encoding="utf-8")
print("Saved outputs to:", OUTPUT_DIR.resolve())


=== REPEATED 5×5 CV SUMMARY — MEAN (SD) ===
      outcome recall_sensitivity  specificity precision_ppv         npv balanced_accuracy f1_positive    accuracy     roc_auc pr_auc_average_precision
      overlap        79.1% (3.7)  54.9% (1.9)   29.1% (1.0) 91.9% (1.3)       67.0% (1.7) 42.5% (1.5) 59.5% (1.4) 74.8% (1.7)              42.2% (3.0)
 perpetration        63.4% (7.0) 57.2% (11.8)   32.0% (4.8) 83.3% (2.8)       60.3% (4.8) 42.2% (4.4) 58.6% (8.2) 65.0% (5.8)              38.0% (5.9)
victimization       60.3% (10.2)  64.9% (8.6)   63.1% (3.3) 63.0% (3.2)       62.6% (2.3) 61.0% (5.8) 62.6% (2.3) 66.8% (3.2)              64.0% (3.3)
=== REPEATED 5×5 CV SUMMARY — MEAN (2.5th–97.5th percentile) ===
      outcome recall_sensitivity       specificity     precision_ppv               npv balanced_accuracy       f1_positive          accuracy           roc_auc pr_auc_average_precision
      overlap  79.1% (74.1–85.7) 54.9% (51.8–58.7) 29.1% (27.4–30.6) 91.9% (90.1–94.1) 67.0% (64.4–69.8

## Suggested manuscript language

Use this only after verifying that the output metrics are coherent.

**Methods/Supplement:**

> As a supplementary stability check, we conducted repeated stratified cross-validation using 5 folds repeated 5 times. In each fold, scaling and PCA were fitted within the training fold only and then applied to the validation fold. This analysis used the final screening-oriented modeling strategy and was intended to assess metric stability across alternative partitions; it did not replace the final held-out evaluation and was not used to reselect the final models.

**Results/Supplement:**

> The repeated cross-validation analysis showed whether the screening-oriented performance pattern was stable across 25 alternative validation folds. Metrics are reported as mean, standard deviation, and empirical 2.5th–97.5th percentile intervals.

**Caution:**

> This repeated CV is an internal stability analysis. It is not external validation and should not be described as prospective or independent validation.
